# Getting started

`GaugeFixer` facilitates interpretation of sequence-function models by removing unconstrained degrees of freedom from the parameter values of one-hot generalized models, such as additive, pairwise and all-order interaction models, and express them in ways that have different interpretations that may be useful for different applications.


> **Note**: The parameters of these models (or the sequence-function map they encode) must be learned before defining a model in `GaugeFixer`.

In this section, we show a simple example for how to define an all-order interaction model and compute the corresponding gauge-fixed coefficients under different gauges using `GaugeFixer`'s efficient algorithms.

### Importing required libraries


In [33]:
import numpy as np
import pandas as pd

from gaugefixer import AllOrderModel

### Defining a sequence-function model

`GaugeFixer` considers sequence-function models for sequences of fixed length `L` with characters drawn from an alphabet. This alphabet can be common to all positions or site specific. However, we will typically consider biological sequences defined over `'dna'`, `'rna` or `'protein'` alphabets. 

For this example, we will define model for 9-nucleotide long RNA sequences for the Shine-Dalgarno sequence landscape inferred by [Martí-Gómez et al. (2026)](https://academic.oup.com/mbe/article/doi/10.1093/molbev/msag023/8456298) from data by [Kuo et al. (2020)](https://genome.cshlp.org/content/30/5/711) as follows:

In [15]:
model = AllOrderModel(L=9, alphabet_name='rna')
model

AllOrderModel(L=9,alphabet_name=rna,n_features=1953125,n_orbits=512)

There are multiple ways of defining the parameters of the model, but the most direct way is by providing the values of the parameters associated to the one-hot features as a `pd.Series` using the `set_params` method. The series must be indexed by the sequence binary features $x_U^u$, each of which must be defined as a tuple containing the set of sites $U$ and the subsequence $u$. Here, we load the set of parameters from a file

In [30]:
from pickle import load
with open('shine_dalgarno.theta.pkl', 'rb') as fhand:
    theta = load(fhand)
theta

((), )                                      0.620094
((0,), A)                                   0.031795
((0,), C)                                  -0.105760
((0,), G)                                   0.104421
((0,), U)                                  -0.030456
                                              ...   
((0, 1, 2, 3, 4, 5, 6, 7, 8), UUUUUUUGU)    0.005219
((0, 1, 2, 3, 4, 5, 6, 7, 8), UUUUUUUUA)   -0.001907
((0, 1, 2, 3, 4, 5, 6, 7, 8), UUUUUUUUC)    0.000431
((0, 1, 2, 3, 4, 5, 6, 7, 8), UUUUUUUUG)    0.001947
((0, 1, 2, 3, 4, 5, 6, 7, 8), UUUUUUUUU)   -0.000471
Length: 1953125, dtype: float64

and use them to set the parameters of the sequence-function model previously defined by simply using `set_params`

In [31]:
model.set_params(theta)


### Fixing the Gauge

Now that the model is completely specified, we may want to interpret the parameter values. However, there are subspaces of parameter values that encode the same sequence-to-function model. Fixing the gauge means choosing one among the many set of parameter values that encode the same model by specifying certain properties of the parameter values e.g. in the `hierarchical` gauge the parameters associated to low order subsequences explain as much variance as possible.

We can do this operation easily by using the method `fix_gauge`. The gauge defined either by their specific names, e.g., `wild-type`, `zero-sum`, or `hierarchical`, or by directly defining the $\lambda$ value as `lda` and the site- and allele-specific probabilities `pi_lc`, which define a specific gauge in the $\lambda-\pi$ family of linear gauges.

For instance, lets express our parameters in the commonly used `zero-sum` gauge.


In [32]:
theta_fixed = model.get_fixed_params(gauge='zero-sum')
theta_fixed

((), )                                      0.620094
((0,), A)                                   0.031795
((0,), C)                                  -0.105760
((0,), G)                                   0.104421
((0,), U)                                  -0.030456
                                              ...   
((0, 1, 2, 3, 4, 5, 6, 7, 8), UUUUUUUGU)    0.005219
((0, 1, 2, 3, 4, 5, 6, 7, 8), UUUUUUUUA)   -0.001907
((0, 1, 2, 3, 4, 5, 6, 7, 8), UUUUUUUUC)    0.000431
((0, 1, 2, 3, 4, 5, 6, 7, 8), UUUUUUUUG)    0.001947
((0, 1, 2, 3, 4, 5, 6, 7, 8), UUUUUUUUU)   -0.000471
Length: 1953125, dtype: float64

Now the parameters can be more easily interpreted. For example, the parameter for the empty subsequence represents the average across all posible sequences 

In [11]:
theta_fixed.loc[[((), '')]]

((), )   -0.039335
dtype: float64

whereas the parameter associated to nucleotide A at position 3 represents the average effect of placing an A at that position across all possible sequence. 

In [12]:
theta_fixed.loc[[((2,), 'A')]]

((2,), A)    0.055764
dtype: float64

But maybe we are interested in the effect of placing an A at position 2 only in sequences with A or G at position one. We can do this easily using the `hierarchical` Gauge with a under a site-independent probability distribution `pi_lc` 

In [13]:
pi_lc = [np.array([0.5, 0, 0.5, 0]), np.array([0.25, 0.25, 0.25, 0.25]), np.array([0.25, 0.25, 0.25, 0.25])]
theta_fixed = model.get_fixed_params(gauge='hierarchical', pi_lc=pi_lc)
theta_fixed.loc[[((2,), "A")]]

((2,), A)    0.026694
dtype: float64